In [ ]:
import geopops
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import starsim as ss

# 1.0 Run Geopops

**GeoPops** is a package for generating geographically and demographically realistic synthetic populations for any US Census location using publically available data. Population generation includes three steps:
1. Generate individuals and households using combinatorial optimization (CO)
2. Assign individuals to school and workplace locations using enrollment data and commute flows
3. Connect individuals within locations using graph algorithms

Resulting files include a list of agents with attributes (e.g., age, gender, race/ethnicity) and networks detailing their connections within home, school, workplace, and group quarters (e.g., correctional facilities, nursing homes) locations. GeoPops is meant to produce reasonable approximations of state and county population characteristics with granularity down to the Census Block Group (CBG).   GeoPops builds on a previous package, [GREASYPOP-CO](https://github.com/CDDEP-DC/GREASYPOP-CO/tree/main) (One Health Trust), and incorporates the following changes:
- All code wrapped in convenient Python package that can be pip installed
- Compatibility with Census data beyond 2019 (in development)
- Automated data downloading
- Users can adjust all config parameters from the front-end
- Class for exporting files compatible with the agent-based modeling software [Starsim](https://starsim.org/) (Institute for Disease Modeling)

## 1.1 Obtain Census API
First, obtain a **Census API key** [here](https://api.census.gov/data/key_signup.html), which will be used for pulling Census data. Getting and activating a key may take a few mintues.

## 1.2 Create Python environment
You'll also need a **Python environment** with the dependencies listed in the GeoPops `pyproject.toml`. I suggest making a virtual environment different from you base environment using python 3.12 or 2.13. I'd also suggest storing it locally so that you never have to re-download package files from the Cloud. Create a local folder where you want to store your GeoPops files. Download the `pyproject.toml` file from this repo and put it in your selected folder folder. In your terminal, navigate to that folder and try the following commands (for Mac). Don't copy the comments; these are just for reference.

    cd "YOUR_FOLDER"                # Navigate to your folder
    python3.12 -m venv geopops-env  # Create a virtual environment with python 3.12 or 3.13
    source geopops-env/bin/activate # Activate the virtual environment
    pip install geopops             # Install geopops and dependencies
    pip install ipykernel           # Install ipykernel so you can find the environment in Jupyter notebooks
    python -m ipykernel install --user --name geopops-env --display-name "Python (geopops-env)"

## 1.3 Create Julia environment

Next, create a **Julia environment** with the dependencies listed below. It may be easiest to store the environment in the same folder you will use for output files. While called with Python commands, combinatorial optimization, school and workplace assignment, and network generation steps occur in Julia to decrease run time. In the future, we plan to have this all in Python. The following terminal commands should work with MacOS. Don't copy the comments; these are just for reference. The Julia [website](https://julialang.org/install/) also has download instructions. If you are using Windows, look for "For Windows instructions, click here".

    cd "YOUR_FOLDER"
    curl -fsSL https://install.julialang.org | sh
    juliaup add 1.9.0        # Install Julia 1.9.0
    juliaup default 1.9.0    # Make 1.9.0 the default (optional)
    julia +1.9.0 --version   # Run that version once
    juliaup update           # Update installed versions
    julia                    # Launch Julia and see version
    Base.active_project()    # Get path where environment is located. Copy this - will need later
    ]                        # Must type. Don't copy paste. Enter package mode.  
                             # Prompt should change to "(@v1.9) pkg>"
    add CSV@0.10.10          # Add required package versions
    add DataFrames@1.5.0
    add Graphs@1.8.0 
    add InlineStrings@1.4.0 
    add JSON@0.21.4
    add StatsBase@0.33.21
    add Distributions@0.25.112
    add MatrixMarket@0.4.0
    add ProportionalFitting@0.3.0
    status                   # View list of packages

Now, you can run the rest of this notebook by selecting the Python environment you just created as the Kernel.

## 1.4 Define parameters
All the parameters to customize your population are stored a file called `config.json`. You can define these parameters in a dictionary which can be passed into `WriteConfig()`. This will write out the `config.json` file with your specified parameters in the output folder you selected. There are a lot of config parameters you can adjust (see `data/config_parameters.csv` for full list). In this notebook, we'll set the year to be 2023 as this is the last year with complete Census data.

| Parameter | Definition |
| -------- | -------- | 
| path | Chose a folder where you want output files to be stored |
| census_api_key | Your Census API key |
| julia_env_path | Path to your Julia environment. Try "/Users/[Name]/.julia/environments/v1.9" or “C:/[YOUR PATH TO JULIA]/.julia/environments/v1.9/Project.toml” |
| main_year | Year of your population, corresponds to year of Census data. main_year=2019 means school year 2019/2020 |
| geos | State or county FIPs codes of your main geography |
| commute_states | State FIPs codes where people commute to and from work (The populaton is "closed" except for commuters) |
| use_pums | State FIPS codes of the PUMS households sampled for combinatorial optimization |

In [ ]:
pars_geopops = {'path': "data", # Set a folder where you want outut files to be stored
                'census_api_key': "YOUR_CENSUS_API_KEY", # Your Census API key
                'julia_env_path': "YOUR_JULIA_ENV_PATH", # Path to your Julia environment, "/Users/Name/.julia/environments/v1.9" or “C:/YOUR PATH TO JULIA/.julia/environments/v1.9/Project.toml” 
                'main_year': 2019, # Year of data
                'geos': ["45083"], # State or county fips of your geographical location of interest.Example of Spartanburg SC
                'commute_states': ["45","37"], # State fips of commute data to download. Example of SC, NC
                'use_pums': ["45","37"], # State fips of PUMS data to download. Example of SC and NC
                } 

c = geopops.WriteConfig(**pars_geopops) # Define parameters for pop generation in config.json
# c.get_pars() # View all parameters from config.json

Updated config.json with new parameters


## 1.5 Download raw data
The `DownloadData()` class pulls raw data files from the internet using the Census API and urls strings. Output files go into the folders census, geo, pums, school, and work. These folders will show up in the output directory you chose as the data are downloaded. `folders_files_tree.rtf` shows how all the files are stored in your output directory. Data are pulled from the following sources:

| Data Source | Role in GeoPops |
|---|---|
| [American Community Survey (ACS)](https://www.census.gov/programs-surveys/acs) | Provides block-group-level demographic and socioeconomic distributions (e.g., age, households, income) that GeoPops matches when constructing the synthetic population. |
| [Decennial Census](https://www.census.gov/programs-surveys/decennial-census) | Used to construct group quarters compositions (institutional vs non‑institutional, civilian vs military) |
| [Public Use Microdata Sample (PUMS)](https://www.census.gov/programs-surveys/acs/microdata.html) | Provides person- and household-level microdata that are reweighted and combined (via combinatorial optimization) to create realistic synthetic households and individuals. |
| [TIGER/Line Shapefiles](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html) | Provide the geographic boundaries (tracts, block groups, etc.) needed to locate agents spatially and to link Census counts to geography. |
| [LEHD LODES (Longitudinal Employer–Household Dynamics)](https://lehd.ces.census.gov/data/#lodes) | Provides origin-destination commute patterns and workplace locations, used to assign workers to jobs and build the work contact network. |
| [County Business Patterns (CBP)](https://www.census.gov/programs-surveys/cbp.html) | Supplies employment counts by industry and geography, used to characterize and weight workplaces in the synthetic population and work network. |
| [Census Tract-to-PUMA Crosswalk](https://www.census.gov/geographies/reference-files/time-series/geo/relationship-files.html) | Links Census tracts to PUMAs so that PUMS microdata (available at the PUMA level) can be aligned with tract/block-group targets. |
| [MCDC Geocorr (2018/2022)](https://mcdc.missouri.edu/applications/geocorr.html) | Provides crosswalks between tracts/CBGs and higher-level geographies (counties, CBSAs, urban-rural), used to attach regional attributes and to aggregate or filter results. |
| [NCES EDGE School Data](https://nces.ed.gov/programs/edge) | Supplies public school locations and characteristics, used to place students and staff and to build realistic school contact networks. |


### Data caveats
GeoPops uses the most recent data available from each source, but the most recent year may not be consistent across all data sources. The following table describes the impact of data year inconsistancies on the synthesized population.

| Data source | Availability | Impact on GeoPops |
|---|---|---|
| ACS | If `main_year >= 2023`, tables `B09019`, `B09020`, and `B09021` fall back to 2022 because newer releases are not available via Census API | Most targets still use requested-year ACS where available, but some household/GQ and age-structure constraints become mixed-year. This can slightly affect employment/GQ balancing and age composition. |
| PUMS | Available through 2023. Password required from Census website for years >=2024 | If `main_year > 2023`, PUMS fallback to 2023 microdata. Household/person samples come from 2023 structure and are reweighted to target-year ACS marginals. |
| LODES | Available through 2023. Not yet published for years >=2024 | If `main_year > 2023`, commute and workplace assignment fallback to 2023 OD flows, so work network patterns reflect 2023 geography/economy. |
| Geocorr | Uses `geocorr2018` for pre-2020 runs; `geocorr2022` for 2020+ runs | Determines CBG↔county/cbsa/urban-rural mapping. Mixed-geography year (2020-2022) can increase missing geo links and reduce fine-scale geographic fidelity. |
| NCES | Available through 2024 | If `main_year > 2024`, school location/enrollment/staff data fallback to the latest available year, so school assignment/network reflect fallback-year school structure. |

In the cell below, set `auto_run=True` to download all data at once, or `auto_run=False` to run one at a time. Runtime can be over 10 minutes on a MacBook Pro with Intel processor and good - but not super high speed - WiFi (some files are big!). Downloading school data especially takes a while so be patient. You may want set your computer not to go to sleep in case this breaks the run.

In [3]:
# Create DownloadData object
d = geopops.DownloadData(auto_run=True)

# Can set auto_run=False to run one function at a time
# d.pull_census_data()
# d.pull_pums_data()
# d.download_shapefiles()
# d.pull_LODES()
# d.download_ct_puma_crosswalk()
# d.geocorr_files()
# d.download_cbp_data()
# d.download_school_data() # This one can take >15 minutes 

Copying geocorr files into geo folder - DownloadData.geocorr_files()
All DownloadData() steps complete


## 1.6 Process data
The `ProcessData()` class reformats all the raw data into interim files that will be used in the next steps in Julia. Intirem files go into the folder processed. Runtime is about 4 minutes with a MacBook Pro with Intel processor. Set auto_run=True to run all at once or auto_run=False to run one at a time. If running one at a time, must be done in order because each function depends on the output of the previous function.

In [4]:
p = geopops.ProcessData(auto_run=True) # auto_run=True to run all 

# Can run one at a time with auto_run=False but must be in order
# because each function depends on the output of the previous one
# p.generate_samples()
# p.generate_gq()
# p.generate_targets()
# p.calc_commute_marginals()
# p.generate_work_sizes()
# p.generate_schools() # Takes a long time

Generating PUMS samples - ProcessData.generate_samples()
Generating group quarters data - ProcessData.generate_gq()
Generating Census targets - ProcessData.generate_targets()
Generating commute marginals - ProcessData.calc_commute_marginals()
Generating employer sizes - ProcessData.generate_work_sizes()
Generating school data - ProcessData.generate_schools()
All ProcessData() steps complete


## 1.6.1 Quality Check
For 2020–2022, some Census products use mixed geography vintages, so small-area targets and PUMA-based microdata are not always perfectly aligned. In practice, this can create partial puma mismatches and force optimization to rely more on county/cbsa/urban-rural fallback instead of local PUMA matching. The data are still usable, but these years should be interpreted with more caution for fine geographic fit. This will make more sense after reading section **1.7.1 Combinatorial optimization** and the **More on CO()** section at the end of this notebook. 

The `geopops.QualityCheck()` class identifies missing geography fields so you can assess the geographical alignment. The `.results` method prints a summary of whether your processed geographies are internally consistent before optimization/export. The `st_puma_overlap` section tells you how many target CBG PUMA codes are actually represented in the sample pool; high overlap means PUMA-level matching is feasible for the mapped targets. The `samp_geo_missingness` section tells you how many sample households are missing cbsa/county/R/U; high missingness means many households cannot be matched at those geography levels and will rely on broader fallback. 

**Take-away:**
Years <=2019 and year 2023 produce robust populations at fine geographic scale (e.g., CBG). Years 2020-2022 can still produce reasonable approximations for aggregate regions (e.g., county) but should be treated with caution when looking at finer geographic scales.

In [5]:
qc = geopops.QualityCheck()
qc.results

QualityCheck summary
- output_dir: data
- main_year: 2019
- st_puma overlap: 2 / 2 (100.00% of target st_puma codes covered)
- samp_geo missing fractions (cbsa/county/R/U): 0.000%, 0.000%, 0.000%, 0.000%
- rows with blank cbsa but county/R/U present: 0
- rows with blank cbsa and county/R/U also blank: 0


{'output_dir': 'data',
 'processed_dir': 'data/processed',
 'main_year': 2019,
 'st_puma_overlap': {'cbg_rows_total': 195,
  'cbg_rows_missing_st_puma': 0,
  'cbg_rows_non_missing_st_puma': 195,
  'cbg_unique_st_puma': 2,
  'samp_unique_st_puma': 108,
  'intersection': 2,
  'cbg_covered_by_samp_fraction': 1.0,
  'cbg_not_in_samp_examples': [],
  'samp_not_in_cbg_examples': ['3700100',
   '3700200',
   '3700300',
   '3700400',
   '3700500',
   '3700600',
   '3700700',
   '3700800',
   '3700900',
   '3701000',
   '3701100',
   '3701201',
   '3701202',
   '3701203',
   '3701204',
   '3701205',
   '3701206',
   '3701207',
   '3701208',
   '3701301'],
  'by_state': {'37': {'cbg_unique_st_puma': 0,
    'samp_unique_st_puma': 78,
    'intersection': 0},
   '45': {'cbg_unique_st_puma': 2,
    'samp_unique_st_puma': 30,
    'intersection': 2}}},
 'samp_geo_missingness': {'missingness': {'cbsa': {'missing_count': 0,
    'missing_fraction': 0.0},
   'county': {'missing_count': 0, 'missing_fractio

## 1.7 Run Julia
The `RunJulia()` class consists of three methods, which can be run separately or all together (sequentially) using `.run_all()`. Each one calls and runs the corresponding Julia scripts. For a more detailed explanation of the methods for each step, see [Tulchinsky et al. 2026](https://pubmed.ncbi.nlm.nih.gov/41762550/).
1. `.CO()` - Combinatorial Optimization
2. `.SynthPop()` - School/workplace assignment and network generation
3. `.Export()` - Exports jlse and to csv

## 1.7.1 Combinatorial optimization
GeoPops uses combinatorial optimization to sample households from the Public Use Microdata Samples (PUMS) until a set is found that reasonably matches overall population characteristics from the American Comunity Survey (ACS) for your geography of interest (defined in geos in `WriteConfig()`).

**PUMS:** Anonymized subset of Census survey responses for regions of around 100,000 people. Contains compositional details of households and members (e.g., the number of five-member households with school-age children).

**ACS:** Geography and demographics for Census areas down to CBG level. Restricted to select summaries of marginal counts (e.g., how many households in a locale have five members and the number of school-age children). Missing household compositional details. 

The distributions of demographic characteristics in a PUMS area may not be representative of the overall population area. CO is a way to combine a set of households from PUMS so that the distributions for demographic characteristics (age, income, gender) of all the individual agents within that set reasonably match ACS data.

See the section **More detail on CO()** at the end of this notebook for further explaination of how `CO()` works and a description of printouts when running.

Runtime for `CO()` is about 5 minutes on a MacBook Pro with Intel processor.



In [ ]:
j = geopops.RunJulia()
j.CO()

## 1.7.2 Assignment and network generation
The `RunJulia.SynthPop()` method performs school and workplace assignment and network generation. Output files go into the folder pop_export. Networks are stored as upper adjacency matrices in .mtx format. 

**Assignment**
| Locations | Methods | Data | Sources |
| -------- | -------- | -------- | -------- |
| Schools | Students assigned to closest school that offer required grade. Teachers assigned to school by selecting teachers from nearby CBGs | Student and staff counts for each grade level in public schools | National Center for Education Statistics (NCES) |
| Workplaces | Iterative proportional fitting (IPF) to make origin-destination commute matrix. IPF to make industry by destination commute matrix for each origin CBG. Includes outside workers | Origin-destination and workplace category matrices | Longitudinal Employer-Household Dynamics (LEHD) |

**Network generation** 
| Network | Algorithm | Parameters |
| -------- | -------- | -------- |
| Household | All agents within a household are fully connected |  |
| School | Stochastic Block Model within each school | Associativity ⍺=0.9. Mean degree K=12. Blocks are grade levels (Pre-k – 12) |
| Workplace | Stochastic Block Model within each workplace | Associativity ⍺=0.9. Mean degree K=8. Blocks are income levels (<40k or ≥ 40k) |
| Group quarters (GQ) | Watts-Strogatz Model within each GQ facility | Mean degree K=12. Rewiring probability β=0.25 |

The Stochastic Block Models are connecting agents within schools so that each agent has a mean of 12 contacts, and ~90% of these contacts are within the same grade. You can adjust the network generation algorithms by changing the following parameters with `WriteConfig.py`:

| Network parameters | Definition |
| -------- | -------- |
| school_associativity_coefficient | School associativity |
| school_K | Mean degree in school network |
| work_associativity_coefficient | Workplace associativity |
| workplace_K | Mean dgree in workplace network |
| gq_K | Mean degree in group quarters network |
| netw_B | rewiring probability for small-world GQ network |

Runtime for `SynthPop()` is about 1.5 minutes.

In [ ]:
j.SynthPop()

## 1.7.3 Export files
The `.Export()` method exports jlse files (creaded during CO() and in the jlse folder) as csv files into the folder pop_export. Runtime is about 30 seconds.

| File| Description |
| -------- | -------- | 
| people_all.csv | Full list of all agents with attributes, including dummy agents that live outside the geo area (e.g., Spartanburg County) |
| people.csv | List of agents with attributes who live inside geo area |
| hh.csv | List of of households with location, PUMs sample index, and household size |
| cbg_idxs.csv | Maps GeoPops cbg_id to 15-digit Census CBG geocode |
| sch_students.csv | List of students by school code |
| sch_workers.csv | List of school workers by school code |
| company_workers.csv | List of agents by employer locations inside geo area, type, and employer id  |
| outside_workers.csv | List of agents that work outside the geo area |
| gq_residents.csv | List of GQ residents by GQ location and GQ id |
| gq_workers.csv | List of GQ workers by location and GQ id |
| gqs.csv | List of GQ facilities with location, type, and GQ id |
| adj_mat_keys.csv | Full list of agents but no attributes |
| adj_dummy_keys.csv | List of agents who live outside geo area and commute in for work, no attributes. Inluded in workplace network |
| adj_out_workers.csv | List of agents who live in area and commute out to work, no attributes. Not in workplace network |
| adj_upper_trang_hh.mtx| Household network upper adjacency matrix |
| adj_upper_trang_sch.mtx| School network upper adjacency matrix |
| adj_upper_trang_wp.mtx| Workplace network upper adjacency matrix |
| adj_upper_trang_gq.mtx| Group quarters network upper adjacency matrix |
| adj_upper_trang_non_hh.mtx| Combination of School, workplace, and gq (NOT "Other" or "Community" network) |

In [8]:
j.Export()

households and people
schools
group quarters
workplaces
done
household adjacency matrix
non-household adjacency matrix
workplace only adjacency matrix
school only adjacency matrix
groups quarters only adjacency matrix
index keys
done


## 1.8 Format for Starsim
Starsim ([Kerr et al. 2024](https://starsim.org/)) is an open-source agent-based modeling software by the Institute for Disease Modeling. It is module-based, allowing the user to customize the disease, contact structure, and population.

The `geopops.ForStarsim()` class has three nested classes for formatting a GeoPops population for Starsim.
* `.People()` creates a Starsim people object of agents and attributes. Stored as `pop_export/starsim/ppl.pkg`
* `.GPNetwork()` transforms the upper adjacency matrices into a Starsim Network. Stored as `pop_export/starsim/net_w.csv`, etc.
* `.SubgroupTracking()` returns a dataframe of outcomes overtime by subgroup. We'll use this later in `5_subgroup_tracking.ipynb`

Files are exported into the pop_export/starsim folder and will be loaded into a Starsim simulation later.
| File for Starsim | Description |
| -------- | -------- | 
| net_g.csv | Grouq quarters network as edge list |
| net_h.csv | Home network as edge list |
| net_s.csv | School network as edge list |
| net_w.csv | Work network as edge list |
| ppl.pkl | Starsim people object of your GeoPops population |


## 1.8.1 People
Create Starsim people object from your GeoPops agents and attributes and view it as a dataframe. The People object is explained in more detail in `2_explore_people.ipynb`.

In [ ]:
ppl = geopops.ForStarsim.People()

## 1.8.2 Make networks
`ForStarsim.GPNetwork()` reformats the upper adjacency matrices (mtx files) for each network into dataframes (also saved as csv files), where `p1` (person 1) is connected to `p2` (person 2) and `edge_weight` is the weight of their connection. The link between the two agents is bi-directional so if either is infected, they could infect the other. With Starsim, edge_weight gets turned into a multiplier in the transmission equation, so increasing edge_weight in one network relative to others would increase transmissions in that network relative to others. More on networks in `3_explore_networks.ipynb`.

In [10]:
h = geopops.ForStarsim.GPNetwork(name='homenet', edge_weight=1.0)
s = geopops.ForStarsim.GPNetwork(name='schoolnet', edge_weight=1.0)
w = geopops.ForStarsim.GPNetwork(name='worknet', edge_weight=1.0)
g = geopops.ForStarsim.GPNetwork(name='gqnet', edge_weight=1.0)

Network csv files created and saved successfully


## Put it all together!
It looks like a lot but when you put it all together you can make a GeoPops population with 6 lines of code!

```
pars_geopops = {'path': "YOUR_OUTPUT_DIR", # Set a folder where you want outut files to be stored
                'census_api_key': "YOUR_CENSUS_API_KEY", # Your Census API key
                'julia_env_path': "YOUR_JULIA_ENV_PATH", # Path to your Julia environment
                'main_year': 2019, # Year of data
                'geos': ["45083"], # Example of Howard County, MD
                'commute_states': ["45", "37"], # State fips of commute data to download
                'use_pums': ["45", "37"]} # State fips of PUMS data to download

 geopops.WriteConfig(**pars_geopops) # Define parameters for pop generation in config.json
 geopops.DownloadData(auto_run=True) # Download raw data
 geopops.ProcessData(auto_run=True) # Preprocess data
 j = geopops.RunJulia() # Run Julia portion
 j.run_all() # Run all methods (CO(), SynthPop(), Export()) in  RunJulia() class
```

## More detail on CO()

CO iteratively adjusts the microdata samples within each geography so that aggregate totals (age, households, etc.) match targets. Each pass focuses on a different geographic level, starting with PUMA (Public use Microdata Area). If a CBG still fails the tolerance criteria after a pass, it is flagged and re‑optimized in subsequent passes at bigger geographic levels (first county, then Core Based Statistical Area (CBSA), then urban/rural) until all geographies fall within the acceptable error threshold or no further improvement is possible. With `WriteConfig()` you can change the error threshold and the number of generations the optimizer runs by redefining the parameters CO_crit_val and CO_maxgens.

Each printout line has three values for a single CBG target:
* First number: the current value of the error metric (E_0) for that optimization run (how far you are, on average, from the target constraints).
* Second number: the number of CBG targets being re‑optimized in that pass (how many CBGs are still “problematic” at this stage).
* Third number: a very small convergence/acceptance value (e.g., likelihood ratio / improvement probability) used internally by the CO algorithm to decide whether the current solution is a meaningful improvement; values close to 0 mean the algorithm is making only tiny improvements and is near convergence.

If a large number of CBGs must be rerun at each geography level and only finally resolve at the urban/rural step, it means:
* The constraints are hard to satisfy at fine spatial scales (PUMA, state, county, CBSA) given the available microdata; many CBGs can’t match their detailed targets without violating others.
* When you move to coarser groupings (urb/rural), the algorithm can trade off errors across CBGs within the same urban/rural class, so it becomes easier to find a configuration where all units fall below the error threshold.
* Practically, this indicates that your population still matches targets well in aggregate, but individual CBGs are being heavily adjusted and some local estimates are more model‑driven than data‑driven.